# Final Performance Benchmark — All 3 Tiers Combined

**Purpose:** Comprehensive E2E performance test after ALL optimization tiers.

## Optimization Summary

### Tier 1 — Quick Wins (commit `4c1464b`)
- **T-OPT-1.1:** Parallel parse batches (`asyncio.gather` + `Semaphore(3)`)
- **T-OPT-1.2:** Cache Vertex AI Search context on retry (skip `retrieve_context()`)
- **T-OPT-1.3:** Skip supervisor on retry (`fail → math_agent`)

### Tier 2 — Medium Optimizations
- **T-OPT-2.1:** Parse-only retry (cache raw text, retry parsing before re-generating)
- **T-OPT-2.3:** Parallel micro-batch generation (split large batches into 2 concurrent calls)
- **T-OPT-2.2:** SKIPPED — Gemini 3 Flash infinite loop with strict JSON schemas

### Tier 3 — Architectural
- **T-OPT-3.2:** SDK migration to `langchain-google-genai` (already done 2026-03-27)
- **T-OPT-3.3:** Deterministic supervisor (`USE_LLM_CLASSIFICATION=False`, skip LLM call)
- **T-OPT-3.4:** Parallel formatter batches (`asyncio.gather` for quiz/flashcard/fill_blank)
- **T-OPT-3.1:** Pre-generate pool — BLOCKED (requires Firestore + Cloud Tasks infrastructure)

## Baseline (pre-optimization, 2026-04-09)

| Difficulty | Time | Iterations |
|---|---|---|
| recall | 125.1s | 1 |
| comprehension | 122.7s | 1 |
| application | 539.9s | 2 |
| high_application | 1290.8s | 4 |
| **TOTAL** | **2078.5s** | - |

## Targets

| Tier | Target |
|---|---|
| Tier 1 | high_application ≤ 800s, total ≤ 1400s |
| Tier 2 | high_application ≤ 400s, total ≤ 900s |
| Combined | 100% delivery (40/40 items) |

## Section 1: Setup & Configuration Verification

In [1]:
import sys, os, time, json
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
os.environ.setdefault('ENV', 'develop')

from src.api.schemas import (
    GenerationRequest, GameType, DifficultyLevel, DocScope, GameContentResponse
)
from src.graph.builder import compile_graph
from src.graph.nodes.math_agent import (
    MICRO_BATCH_THRESHOLD, MAX_PARSE_ONLY_RETRIES,
    MAX_GENERATION_RETRIES, MAX_PARSE_RETRIES, PARSE_BATCH_SIZE,
)
from src.graph.nodes.supervisor import USE_LLM_CLASSIFICATION
from src.config.constants import FORMATTER_BATCH_SIZE

print('=' * 60)
print('CONFIGURATION VERIFICATION')
print('=' * 60)
print(f'Tier 1:')
print(f'  PARSE_BATCH_SIZE = {PARSE_BATCH_SIZE}')
print(f'  MAX_PARSE_RETRIES = {MAX_PARSE_RETRIES}')
print(f'Tier 2:')
print(f'  MAX_PARSE_ONLY_RETRIES = {MAX_PARSE_ONLY_RETRIES}')
print(f'  MAX_GENERATION_RETRIES = {MAX_GENERATION_RETRIES}')
print(f'  MICRO_BATCH_THRESHOLD = {MICRO_BATCH_THRESHOLD}')
print(f'Tier 3:')
print(f'  USE_LLM_CLASSIFICATION = {USE_LLM_CLASSIFICATION}')
print(f'  FORMATTER_BATCH_SIZE = {FORMATTER_BATCH_SIZE}')
print(f'\nAll imports OK')


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CONFIGURATION VERIFICATION
Tier 1:
  PARSE_BATCH_SIZE = 7
  MAX_PARSE_RETRIES = 2
Tier 2:
  MAX_PARSE_ONLY_RETRIES = 2
  MAX_GENERATION_RETRIES = 2
  MICRO_BATCH_THRESHOLD = 20
Tier 3:
  USE_LLM_CLASSIFICATION = False
  FORMATTER_BATCH_SIZE = 7

All imports OK


## Section 2: SDK & Architecture Verification

In [2]:
import inspect
import src.services.llm as llm_mod
import src.graph.nodes.supervisor as sup_mod
import src.graph.nodes.formatter as fmt_mod

# T-OPT-3.2: Verify SDK migration
llm_src = inspect.getsource(llm_mod)
assert 'ChatGoogleGenerativeAI' in llm_src, 'Expected ChatGoogleGenerativeAI in llm.py'
assert 'ChatVertexAI' not in llm_src, 'ChatVertexAI should not be in llm.py'
print('T-OPT-3.2: ChatGoogleGenerativeAI in use, no ChatVertexAI')

# T-OPT-3.3: Verify deterministic supervisor
assert not USE_LLM_CLASSIFICATION, 'Expected USE_LLM_CLASSIFICATION=False'
print(f'T-OPT-3.3: USE_LLM_CLASSIFICATION = {USE_LLM_CLASSIFICATION}')

# T-OPT-3.4: Verify parallel formatter (asyncio.gather in format functions)
fmt_src = inspect.getsource(fmt_mod)
gather_count = fmt_src.count('asyncio.gather')
assert gather_count >= 3, f'Expected >= 3 asyncio.gather calls in formatter, got {gather_count}'
print(f'T-OPT-3.4: {gather_count} asyncio.gather calls in formatter (quiz/flashcard/fill_blank)')

# T-OPT-1.3: Verify graph topology
from src.graph.builder import compile_graph
graph = compile_graph()
print(f'Graph compiled successfully')

print('\nAll architecture checks passed')


T-OPT-3.2: ChatGoogleGenerativeAI in use, no ChatVertexAI
T-OPT-3.3: USE_LLM_CLASSIFICATION = False
T-OPT-3.4: 7 asyncio.gather calls in formatter (quiz/flashcard/fill_blank)
2026-04-11 09:29:44 [info     ] creating_graph                
2026-04-11 09:29:44 [info     ] graph_created                 
2026-04-11 09:29:44 [info     ] graph_compiled_without_checkpointer
Graph compiled successfully

All architecture checks passed


## Section 3: E2E Runner

In [3]:
NUM_QUESTIONS = 10
TOPIC = "Đạo hàm và ứng dụng"

# Baseline numbers (pre-optimization, E2E 2026-04-09)
BASELINE = {
    "recall":           {"time": 125.1, "iterations": 1},
    "comprehension":    {"time": 122.7, "iterations": 1},
    "application":      {"time": 539.9, "iterations": 2},
    "high_application": {"time": 1290.8, "iterations": 4},
}
BASELINE_TOTAL = sum(b["time"] for b in BASELINE.values())

# Performance targets
TARGETS = {
    "tier1": {"ha_max": 800, "total_max": 1400},
    "tier2": {"ha_max": 400, "total_max": 900},
}


async def run_e2e(difficulty: str):
    """Run pipeline end-to-end and return (num_delivered, elapsed, iterations, errors)."""
    req = GenerationRequest(
        user_id="perf_benchmark_final",
        topic=TOPIC,
        game_types=[GameType.QUIZ],
        num_questions=NUM_QUESTIONS,
        difficulty=DifficultyLevel(difficulty),
        doc_scope=DocScope.SYSTEM,
    )
    app = compile_graph()
    initial_state = {
        "request": req,
        "doc_scope": req.doc_scope.value,
        "iteration_count": 0,
        "rejected_items": [],
        "reviewed_items": [],
        "errors": [],
    }
    t0 = time.time()
    result = await app.ainvoke(initial_state)
    elapsed = time.time() - t0

    output: GameContentResponse | None = result.get("final_output")
    delivered = len(output.content.quiz) if output else 0
    iterations = result.get("iteration_count", 0)
    errors = result.get("errors", [])
    return delivered, elapsed, iterations, errors


print(f"Runner ready — Topic: {TOPIC} | {NUM_QUESTIONS} questions per level")
print(f"Baseline total: {BASELINE_TOTAL:.1f}s")

Runner ready — Topic: Đạo hàm và ứng dụng | 10 questions per level
Baseline total: 2078.5s


## Section 4: Run Full Benchmark (All 4 Difficulty Levels)

In [4]:
print("=" * 80)
print("FINAL PERFORMANCE BENCHMARK — ALL TIERS COMBINED")
print(f"Topic: {TOPIC} | Requested: {NUM_QUESTIONS} quiz per level")
print("=" * 80)

e2e_results = {}
total_start = time.time()

for diff in ["recall", "comprehension", "application", "high_application"]:
    print(f"\n--- Running {diff} ---")
    delivered, elapsed, iters, errors = await run_e2e(diff)
    pct = delivered / NUM_QUESTIONS * 100
    status = "PASS" if delivered >= NUM_QUESTIONS else "FAIL"
    baseline_t = BASELINE[diff]["time"]
    speedup = (baseline_t - elapsed) / baseline_t * 100
    print(f"  [{status}] Delivered: {delivered}/{NUM_QUESTIONS} ({pct:.0f}%) in {elapsed:.1f}s ({iters} iter)")
    print(f"  Baseline: {baseline_t:.1f}s -> Now: {elapsed:.1f}s ({speedup:+.1f}%)")
    if errors:
        print(f"  Errors: {errors}")
    e2e_results[diff] = {
        "delivered": delivered,
        "requested": NUM_QUESTIONS,
        "pct": pct,
        "elapsed": elapsed,
        "iterations": iters,
        "passed": delivered >= NUM_QUESTIONS,
        "baseline": baseline_t,
        "speedup_pct": speedup,
    }

wall_clock = time.time() - total_start
print(f"\n--- Wall clock: {wall_clock:.1f}s ---")


FINAL PERFORMANCE BENCHMARK — ALL TIERS COMBINED
Topic: Đạo hàm và ứng dụng | Requested: 10 quiz per level

--- Running recall ---
2026-04-11 09:29:54 [info     ] creating_graph                
2026-04-11 09:29:54 [info     ] graph_created                 
2026-04-11 09:29:54 [info     ] graph_compiled_without_checkpointer
2026-04-11 09:29:54 [info     ] supervisor_analyzing           doc_scope=system topic='Đạo hàm và ứng dụng' user_id=perf_benchmark_final
2026-04-11 09:29:54 [info     ] supervisor_deterministic       content_type=math reason='Phase 1 — USE_LLM_CLASSIFICATION=False'
2026-04-11 09:29:54 [info     ] supervisor_classified          content_type=math
2026-04-11 09:29:54 [info     ] routing_to_agent               content_type=math route=math_agent
2026-04-11 09:29:54 [info     ] math_agent_starting            doc_scope=system iteration=0 num_questions=10 num_still_needed=10 num_to_generate=13 topic='Đạo hàm và ứng dụng' user_id=perf_benchmark_final
2026-04-11 09:29:54 [info

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-04-11 09:29:56 [debug    ] retrieved_documents            count=22
2026-04-11 09:29:56 [info     ] context_retrieved              chunks=22 sources=22
2026-04-11 09:29:56 [info     ] math_agent_context_retrieved   context_chunks=22
2026-04-11 09:29:56 [info     ] creating_generation_llm        location=global max_output_tokens=8192 model=gemini-3-flash-preview temperature=0.7
2026-04-11 09:29:56 [info     ] rate_limiter_init_bucket       capacity=5.0 refill_rate=1.0 rpm=60
2026-04-11 09:29:56 [info     ] rate_limiter_init_semaphore    max_concurrent=10
2026-04-11 09:30:13 [info     ] math_agent_code_execution_done batch_size=13 code_traces_count=2 generation_attempt=1 text_length=3497
2026-04-11 09:30:13 [info     ] creating_generation_llm        location=global max_output_tokens=8192 model=gemini-3-flash-preview temperature=0.3


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-04-11 09:30:25 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=6
2026-04-11 09:30:29 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-04-11 09:30:29 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=13 parse_attempt=1
2026-04-11 09:30:29 [info     ] math_agent_generated           items_count=13
2026-04-11 09:30:29 [info     ] reviewer_reviewing             items_count=13 iteration=0
2026-04-11 09:30:29 [info     ] creating_review_llm            location=global max_output_tokens=4096 model=gemini-3-flash-preview temperature=0.0
2026-04-11 09:30:41 [info     ] reviewer_completed             iteration=0 passed=13 rejected=0
2026-04-11 09:30:41 [info     ] review_router_deciding         iteration=1 max_iterations=5 pass_rate=1.0 rejected=0 reviewed=13
2026-04-11 09:30:41 [info     ] review_router_pass             reason=sufficient_items
2026-04-11 09:30:41 [info     ] formatter_trimming             after=10 before=13
2026

...

2026-04-11 09:33:29 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-04-11 09:33:43 [warning  ] math_agent_batch_parse_exception attempt=1 batch=2 error='Failed to parse ContentItemList from completion {"items": [{"question": "Tìm $m$ để hàm số $y = x^3 - 3mx^2 + 3(m^2-1)x$ đạt cực đại tại $x = 1$.", "answer": "$m = 2$", "explanation": "Ta có $y\' = 3x^2 - 6mx + 3(m^2-1)$. Hàm số đạt cực trị tại $x=1$ thì $y\'(1) = 0 \\\\Leftrightarrow 3 - 6m + 3m^2 - 3 = 0 \\\\Leftrightarrow 3m^2 - 6m = 0 \\\\Leftrightarrow m=0$ hoặc $m=2$. Với $m=0, y\'\' = 6x \\\\Rightarrow y\'\'(1) = 6 > 0$ (Cực tiểu - Loại). Với $m=2, y\'\' = 6x - 12 \\\\Rightarrow y\'\'(1) = -6 < 0$ (Cực đại - Thỏa mãn).", "topic": "Tham số m - Cực trị", "computation_trace": null}, {"question": "Tìm tọa độ điểm cực tiểu của hàm số $y = e^x - 2x$.", "answer": "$(\\\\ln 2; 2 - 2\\\\ln 2)$", "explanation": "Ta có $y\' = e^x - 2$. $y\' = 0 \\\\Leftrightarrow e^x = 2 \\\\Leftrightarrow x = \\\\ln 2$. Vì $y\'\'

/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_community/vertex_ai_search.py:417: UserWarning: Beta features are configured but beta=False. The following beta features will be ignored:['custom_embedding_ratio']
  warnings.warn(


2026-04-11 09:35:48 [debug    ] retrieved_documents            count=13
2026-04-11 09:35:48 [info     ] context_retrieved              chunks=13 sources=13
2026-04-11 09:35:48 [info     ] math_agent_context_retrieved   context_chunks=13
2026-04-11 09:35:48 [info     ] math_agent_micro_batch_generation batch_a=12 batch_b=13 total=25
2026-04-11 09:36:24 [info     ] math_agent_code_execution_done batch_size=12 code_traces_count=2 generation_attempt=1 text_length=7937


/home/sakana/miniconda3/envs/AIservice/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:1076: UserWarning: 
        - 'executable_code': Always present.
        - 'execution_result' & 'image_url': May be absent for some queries.

        Validate before using in production.

  warnings.warn(


2026-04-11 09:36:44 [info     ] math_agent_code_execution_done batch_size=13 code_traces_count=6 generation_attempt=1 text_length=7801
2026-04-11 09:36:53 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-04-11 09:37:06 [info     ] math_agent_batch_parsed        attempt=1 batch=1 items=7
2026-04-11 09:37:07 [info     ] math_agent_batch_parsed        attempt=1 batch=2 items=6
2026-04-11 09:37:07 [info     ] math_agent_parse_succeeded     generation_attempt=1 items=13 parse_attempt=1
2026-04-11 09:37:12 [warning  ] math_agent_batch_parse_exception attempt=1 batch=2 error='Failed to parse ContentItemList from completion {"items": [{"question": "Cho hàm số $f(x)$ có đạo hàm $f\'(x) = (x-1)^2(x^2-2x)$. Có bao nhiêu giá trị nguyên của tham số $m$ để hàm số $g(x) = f(x^2 - 8x + m)$ có đúng 5 điểm cực trị?", "answer": "m ∈ {1, 2, 3, ... }"}]}. Got: 2 validation errors for ContentItemList\nitems.0.explanation\n  Field required [type=missing, input_value={\'question\': "C

## Section 5: Comprehensive Verdict

In [5]:
print("\n" + "=" * 95)
print("FINAL PERFORMANCE BENCHMARK — COMPREHENSIVE RESULTS")
print("=" * 95)
print(f"{'Difficulty':<20} {'Delivered':>9} {'Rate':>7} {'Baseline':>9} {'Now':>8} {'Speedup':>9} {'Iters':>6} {'Status':>8}")
print("-" * 95)

all_passed = True
for diff, r in e2e_results.items():
    status = "PASS" if r["passed"] else "FAIL"
    if not r["passed"]:
        all_passed = False
    print(
        f"{diff:<20} {r['delivered']:>5}/{r['requested']:<3} {r['pct']:>6.0f}% "
        f"{r['baseline']:>8.1f}s {r['elapsed']:>7.1f}s {r['speedup_pct']:>+8.1f}% "
        f"{r['iterations']:>5} {status:>8}"
    )

print("-" * 95)
total_d = sum(r["delivered"] for r in e2e_results.values())
total_r = sum(r["requested"] for r in e2e_results.values())
total_pct = total_d / total_r * 100 if total_r else 0
total_time = sum(r["elapsed"] for r in e2e_results.values())
total_speedup = (BASELINE_TOTAL - total_time) / BASELINE_TOTAL * 100
total_iters = sum(r["iterations"] for r in e2e_results.values())
print(
    f"{'TOTAL':<20} {total_d:>5}/{total_r:<3} {total_pct:>6.0f}% "
    f"{BASELINE_TOTAL:>8.1f}s {total_time:>7.1f}s {total_speedup:>+8.1f}% "
    f"{total_iters:>5}"
)

# --- Target Verification ---
ha_time = e2e_results["high_application"]["elapsed"]

print("\n" + "=" * 95)
print("PERFORMANCE TARGETS")
print("=" * 95)

# Delivery
delivery_ok = all_passed
print(f"  {'[OK]' if delivery_ok else '[FAIL]'} 100% delivery: {total_d}/{total_r} ({total_pct:.0f}%)")

# Tier 1 targets
t1_ha = TARGETS['tier1']['ha_max']
t1_total = TARGETS['tier1']['total_max']
t1_ha_ok = ha_time <= t1_ha
t1_total_ok = total_time <= t1_total
print(f"  {'[OK]' if t1_ha_ok else '[WARN]'} [Tier 1] high_application <= {t1_ha}s: {ha_time:.1f}s")
print(f"  {'[OK]' if t1_total_ok else '[WARN]'} [Tier 1] Total <= {t1_total}s: {total_time:.1f}s")

# Tier 2 targets
t2_ha = TARGETS['tier2']['ha_max']
t2_total = TARGETS['tier2']['total_max']
t2_ha_ok = ha_time <= t2_ha
t2_total_ok = total_time <= t2_total
print(f"  {'[OK]' if t2_ha_ok else '[WARN]'} [Tier 2] high_application <= {t2_ha}s: {ha_time:.1f}s")
print(f"  {'[OK]' if t2_total_ok else '[WARN]'} [Tier 2] Total <= {t2_total}s: {total_time:.1f}s")

# --- Optimization Breakdown ---
print("\n" + "=" * 95)
print("OPTIMIZATION BREAKDOWN")
print("=" * 95)
print("  Tier 1 (Quick Wins):")
print(f"    T-OPT-1.1: Parallel parse batches (PARSE_BATCH_SIZE={PARSE_BATCH_SIZE})")
print(f"    T-OPT-1.2: Cache Vertex AI Search on retry")
print(f"    T-OPT-1.3: Skip supervisor on retry (fail -> math_agent)")
print("  Tier 2 (Medium):")
print(f"    T-OPT-2.1: Parse-only retry (MAX_PARSE_ONLY_RETRIES={MAX_PARSE_ONLY_RETRIES})")
print(f"    T-OPT-2.3: Parallel micro-batch (MICRO_BATCH_THRESHOLD={MICRO_BATCH_THRESHOLD})")
print(f"    T-OPT-2.2: SKIPPED (Gemini 3 Flash JSON mode infinite loop)")
print("  Tier 3 (Architectural):")
print(f"    T-OPT-3.2: SDK migration — ChatGoogleGenerativeAI")
print(f"    T-OPT-3.3: Deterministic supervisor (USE_LLM_CLASSIFICATION={USE_LLM_CLASSIFICATION})")
print(f"    T-OPT-3.4: Parallel formatter batches (FORMATTER_BATCH_SIZE={FORMATTER_BATCH_SIZE})")
print(f"    T-OPT-3.1: Pre-gen pool — BLOCKED (needs infrastructure)")

# --- Final Verdict ---
print("\n" + "=" * 95)
if delivery_ok:
    print(f"100% delivery maintained | Total speedup: {total_speedup:+.1f}% ({BASELINE_TOTAL:.0f}s -> {total_time:.0f}s)")
else:
    failed = [d for d, r in e2e_results.items() if not r["passed"]]
    print(f"DELIVERY REGRESSION — Failed: {', '.join(failed)}")
print("=" * 95)



FINAL PERFORMANCE BENCHMARK — COMPREHENSIVE RESULTS
Difficulty           Delivered    Rate  Baseline      Now   Speedup  Iters   Status
-----------------------------------------------------------------------------------------------
recall                  10/10     100%    125.1s    59.6s    +52.4%     1     PASS
comprehension           10/10     100%    122.7s    99.8s    +18.6%     1     PASS
application             10/10     100%    539.9s   192.8s    +64.3%     2     PASS
high_application        10/10     100%   1290.8s   156.2s    +87.9%     1     PASS
-----------------------------------------------------------------------------------------------
TOTAL                   40/40     100%   2078.5s   508.4s    +75.5%     5

PERFORMANCE TARGETS
  [OK] 100% delivery: 40/40 (100%)
  [OK] [Tier 1] high_application <= 800s: 156.2s
  [OK] [Tier 1] Total <= 1400s: 508.4s
  [OK] [Tier 2] high_application <= 400s: 156.2s
  [OK] [Tier 2] Total <= 900s: 508.4s

OPTIMIZATION BREAKDOWN
  Tier 1 (

## Section 6: Export Results (Optional)

In [6]:
# Save results to JSON for tracking
from datetime import datetime

report = {
    "timestamp": datetime.now().isoformat(),
    "topic": TOPIC,
    "num_questions": NUM_QUESTIONS,
    "tiers": ["tier1", "tier2", "tier3"],
    "baseline_total_s": BASELINE_TOTAL,
    "optimized_total_s": round(total_time, 1),
    "speedup_pct": round(total_speedup, 1),
    "delivery_rate": f"{total_d}/{total_r}",
    "all_passed": all_passed,
    "per_difficulty": {
        diff: {
            "delivered": r["delivered"],
            "elapsed_s": round(r["elapsed"], 1),
            "baseline_s": r["baseline"],
            "speedup_pct": round(r["speedup_pct"], 1),
            "iterations": r["iterations"],
        }
        for diff, r in e2e_results.items()
    },
    "config": {
        "PARSE_BATCH_SIZE": PARSE_BATCH_SIZE,
        "MAX_PARSE_ONLY_RETRIES": MAX_PARSE_ONLY_RETRIES,
        "MAX_GENERATION_RETRIES": MAX_GENERATION_RETRIES,
        "MICRO_BATCH_THRESHOLD": MICRO_BATCH_THRESHOLD,
        "USE_LLM_CLASSIFICATION": USE_LLM_CLASSIFICATION,
        "FORMATTER_BATCH_SIZE": FORMATTER_BATCH_SIZE,
    },
}

print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "timestamp": "2026-04-11T09:38:37.874805",
  "topic": "Đạo hàm và ứng dụng",
  "num_questions": 10,
  "tiers": [
    "tier1",
    "tier2",
    "tier3"
  ],
  "baseline_total_s": 2078.5,
  "optimized_total_s": 508.4,
  "speedup_pct": 75.5,
  "delivery_rate": "40/40",
  "all_passed": true,
  "per_difficulty": {
    "recall": {
      "delivered": 10,
      "elapsed_s": 59.6,
      "baseline_s": 125.1,
      "speedup_pct": 52.4,
      "iterations": 1
    },
    "comprehension": {
      "delivered": 10,
      "elapsed_s": 99.8,
      "baseline_s": 122.7,
      "speedup_pct": 18.6,
      "iterations": 1
    },
    "application": {
      "delivered": 10,
      "elapsed_s": 192.8,
      "baseline_s": 539.9,
      "speedup_pct": 64.3,
      "iterations": 2
    },
    "high_application": {
      "delivered": 10,
      "elapsed_s": 156.2,
      "baseline_s": 1290.8,
      "speedup_pct": 87.9,
      "iterations": 1
    }
  },
  "config": {
    "PARSE_BATCH_SIZE": 7,
    "MAX_PARSE_ONLY_RETRIES

In [7]:
# ROOT CAUSE ANALYSIS - Why recall & comprehension regressed
import math
from src.config.constants import OVERSHOOT_BY_DIFFICULTY, YIELD_OVERSHOOT_RATIO
from src.graph.nodes.math_agent import MICRO_BATCH_THRESHOLD

print("=== MICRO-BATCH THRESHOLD ANALYSIS ===")
print(f"MICRO_BATCH_THRESHOLD = {MICRO_BATCH_THRESHOLD}")
print()

for diff in ["recall", "comprehension", "application", "high_application"]:
    base = OVERSHOOT_BY_DIFFICULTY.get(diff, YIELD_OVERSHOOT_RATIO)
    overshoot = base * (1 + 0.3 * 0)  # iteration 0
    num_needed = 10
    num_gen = math.ceil(num_needed * overshoot)
    num_gen = max(num_gen, num_needed + 3)  # C5 floor
    triggers_micro = num_gen > MICRO_BATCH_THRESHOLD
    half = num_gen // 2
    rest = num_gen - half

    r = e2e_results[diff]
    baseline_iters = BASELINE[diff]["iterations"]
    delta_iters = r["iterations"] - baseline_iters

    print(f"{diff}:")
    print(f"  overshoot={base}, num_to_generate={num_gen}, C5_floor={num_needed+3}")
    if triggers_micro:
        print(f"  micro-batch: YES -> split {half}+{rest}")
    else:
        print(f"  micro-batch: NO -> single batch")
    print(f"  RESULT: {r['iterations']} iters, {r['elapsed']:.1f}s")
    print(f"  BASELINE: {baseline_iters} iter, {BASELINE[diff]['time']:.1f}s")
    if delta_iters > 0:
        tag = "MICRO-BATCH SUSPECT" if triggers_micro else "LLM nondeterminism"
        print(f"  WARNING: +{delta_iters} extra iteration(s) -- {tag}")
    print()

=== MICRO-BATCH THRESHOLD ANALYSIS ===
MICRO_BATCH_THRESHOLD = 20

recall:
  overshoot=1.1, num_to_generate=13, C5_floor=13
  micro-batch: NO -> single batch
  RESULT: 1 iters, 59.6s
  BASELINE: 1 iter, 125.1s

comprehension:
  overshoot=1.2, num_to_generate=13, C5_floor=13
  micro-batch: NO -> single batch
  RESULT: 1 iters, 99.8s
  BASELINE: 1 iter, 122.7s

application:
  overshoot=1.4, num_to_generate=14, C5_floor=13
  micro-batch: NO -> single batch
  RESULT: 2 iters, 192.8s
  BASELINE: 2 iter, 539.9s

high_application:
  overshoot=2.5, num_to_generate=25, C5_floor=13
  micro-batch: YES -> split 12+13
  RESULT: 1 iters, 156.2s
  BASELINE: 4 iter, 1290.8s

